In [ ]:
# ==========================================
# CONFORMAL CALIBRATION
# ==========================================

history = nn_out["history"]
horizon = nn_out["horizon"]

y_log = np.log1p(series)

X_all, Y_all, origins_all = make_sliding_windows(
    y_log,
    history,
    horizon
)

n = len(X_all)

train_end = int(n * 0.60)
cal_end = int(n * 0.80)

X_cal = X_all[train_end:cal_end]
Y_cal = Y_all[train_end:cal_end]

X_test = X_all[cal_end:]
Y_test = Y_all[cal_end:]

print("\nCalibration windows:", len(X_cal))
print("Test windows:", len(X_test))

In [ ]:
# ==========================================
# SCALE USING TRAINING STATISTICS
# ==========================================

mean = nn_out["mean"]
std = nn_out["std"]

X_cal_norm = (
    X_cal - mean
) / std

X_test_norm = (
    X_test - mean
) / std

In [ ]:
# ==========================================
# CALIBRATION PREDICTIONS
# ==========================================

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

model = nn_out["model"]

X_cal_t = torch.tensor(
    X_cal_norm,
    dtype=torch.float32
).to(device)

with torch.no_grad():

    lower_cal, upper_cal = model(
        X_cal_t
    )

lower_cal = (
    lower_cal.cpu().numpy()
    * std + mean
)

upper_cal = (
    upper_cal.cpu().numpy()
    * std + mean
)

In [ ]:
# ==========================================
# CONFORMAL SCORES
# ==========================================

cal_scores = np.maximum(
    lower_cal - Y_cal,
    Y_cal - upper_cal
)

print("Calibration score shape:")
print(cal_scores.shape)

In [ ]:
# ==========================================
# HORIZON-SPECIFIC QHAT
# ==========================================

alpha = 0.10

qhat_per_horizon = []

for h in range(horizon):

    scores_h = cal_scores[:, h]

    qhat_h = np.quantile(
        scores_h,
        1 - alpha,
        method="higher"
    )

    qhat_per_horizon.append(
        qhat_h
    )

qhat_per_horizon = np.array(
    qhat_per_horizon
)

print("\nConformal corrections:")
print(qhat_per_horizon)

In [ ]:
# ==========================================
# TEST PREDICTIONS
# ==========================================

X_test_t = torch.tensor(
    X_test_norm,
    dtype=torch.float32
).to(device)

with torch.no_grad():

    lower_test_raw, upper_test_raw = model(
        X_test_t
    )

lower_test_raw = (
    lower_test_raw.cpu().numpy()
    * std + mean
)

upper_test_raw = (
    upper_test_raw.cpu().numpy()
    * std + mean
)

In [ ]:
# ==========================================
# CONFORMALIZED INTERVALS
# ==========================================

lower_test_conf = (
    lower_test_raw
    - qhat_per_horizon
)

upper_test_conf = (
    upper_test_raw
    + qhat_per_horizon
)

In [ ]:
# ==========================================
# ORIGINAL INTERVALS
# ==========================================

rmse_raw, coverage_raw = evaluate_quantile_forecasts(
    Y_test,
    lower_test_raw,
    upper_test_raw
)

winkler_raw = compute_winkler_arrays(
    Y_test,
    lower_test_raw,
    upper_test_raw
)

# ==========================================
# CONFORMAL INTERVALS
# ==========================================

rmse_conf, coverage_conf = evaluate_quantile_forecasts(
    Y_test,
    lower_test_conf,
    upper_test_conf
)

winkler_conf = compute_winkler_arrays(
    Y_test,
    lower_test_conf,
    upper_test_conf
)

print("\n========================")
print("BEFORE CONFORMAL")
print("========================")
print("Coverage :", coverage_raw)
print("Winkler  :", winkler_raw)

print("\n========================")
print("AFTER CONFORMAL")
print("========================")
print("Coverage :", coverage_conf)
print("Winkler  :", winkler_conf)

In [ ]:
conformal_results = {

    "Y_test": Y_test,

    "lower_raw": lower_test_raw,
    "upper_raw": upper_test_raw,

    "lower_conf": lower_test_conf,
    "upper_conf": upper_test_conf,

    "qhat_per_horizon": qhat_per_horizon,

    "coverage_raw": coverage_raw,
    "coverage_conf": coverage_conf,

    "winkler_raw": winkler_raw,
    "winkler_conf": winkler_conf
}

In [ ]:
idx = 0

actual = conformal_results["Y_test"][idx]

lower_raw = conformal_results["lower_raw"][idx]
upper_raw = conformal_results["upper_raw"][idx]

lower_conf = conformal_results["lower_conf"][idx]
upper_conf = conformal_results["upper_conf"][idx]

pred = (lower_raw + upper_raw) / 2

plt.figure(figsize=(12,5))

plt.plot(actual, label="Actual", color="black")
plt.plot(pred, label="Forecast", color="red")

plt.fill_between(
    np.arange(len(actual)),
    lower_raw,
    upper_raw,
    alpha=0.3,
    label="Original PI"
)

plt.fill_between(
    np.arange(len(actual)),
    lower_conf,
    upper_conf,
    alpha=0.2,
    label="Conformal PI"
)

plt.legend()
plt.title("Original vs Conformalized Prediction Interval")
plt.show()

In [ ]:
conformal_window_metrics = []

for i in range(len(Y_test)):

    y_true = Y_test[i]

    lower = lower_test_conf[i]
    upper = upper_test_conf[i]

    pred = (lower + upper) / 2

    rmse = np.sqrt(
        np.mean((y_true - pred) ** 2)
    )

    coverage = np.mean(
        (y_true >= lower)
        &
        (y_true <= upper)
    )

    winkler = compute_winkler_arrays(
        y_true,
        lower,
        upper
    )

    conformal_window_metrics.append({
        "idx": i,
        "rmse": rmse,
        "coverage": coverage,
        "winkler": winkler
    })

best_conf_idx = min(
    conformal_window_metrics,
    key=lambda x: x["rmse"]
)["idx"]

worst_conf_idx = max(
    conformal_window_metrics,
    key=lambda x: x["rmse"]
)["idx"]

print("Best conformal window :", best_conf_idx)
print("Worst conformal window:", worst_conf_idx)

In [ ]:
origins_test_conf = origins_all[cal_end:]

In [ ]:
def plot_conformal_window(
    subset,
    X_test,
    Y_test,
    lower_conf,
    upper_conf,
    origins_test,
    idx,
    title
):

    history = X_test[idx]

    actual = Y_test[idx]

    lower = lower_conf[idx]
    upper = upper_conf[idx]

    pred = (lower + upper) / 2

    origin = int(origins_test[idx])

    history_len = len(history)
    horizon_len = len(actual)

    history_dates = subset.index[
        origin-history_len:origin
    ]

    future_dates = subset.index[
        origin:origin+horizon_len
    ]

    rmse = np.sqrt(
        np.mean((actual - pred) ** 2)
    )

    coverage = np.mean(
        (actual >= lower)
        &
        (actual <= upper)
    )

    winkler = compute_winkler_arrays(
        actual,
        lower,
        upper
    )

    plt.figure(figsize=(14,6))

    plt.plot(
        history_dates,
        history,
        color="black",
        linewidth=2,
        label="History"
    )

    plt.plot(
        future_dates,
        actual,
        color="blue",
        linewidth=2,
        label="Actual"
    )

    plt.plot(
        future_dates,
        pred,
        color="red",
        linewidth=2,
        label="Forecast"
    )

    plt.fill_between(
        future_dates,
        lower,
        upper,
        color="green",
        alpha=0.25,
        label="Conformal Interval"
    )

    plt.axvline(
        history_dates[-1],
        color="red",
        linestyle="--",
        linewidth=2
    )

    plt.title(
        f"{title}\n"
        f"RMSE={rmse:.3f} | "
        f"Coverage={coverage:.3f} | "
        f"Winkler={winkler:.3f}"
    )

    plt.legend()

    plt.xticks(rotation=45)

    plt.tight_layout()

    plt.show()

In [ ]:
plot_conformal_window(
    subset=subset,
    X_test=X_test,
    Y_test=Y_test,
    lower_conf=lower_test_conf,
    upper_conf=upper_test_conf,
    origins_test=origins_test_conf,
    idx=best_conf_idx,
    title="Best Conformal Forecast Window"
)

In [ ]:
plot_conformal_window(
    subset=subset,
    X_test=X_test,
    Y_test=Y_test,
    lower_conf=lower_test_conf,
    upper_conf=upper_test_conf,
    origins_test=origins_test_conf,
    idx=worst_conf_idx,
    title="Worst Conformal Forecast Window"
)